ASTR8004 Assignment 2, Question 2

2.1 ADQL Query. Was unable to compute the join in the query, so this is as two separate parts, and joined together in python

In [23]:
from astroquery.gaia import Gaia

query_stage1 = """
SELECT
    gaia_filtered.source_id, gaia_filtered.ra, gaia_filtered.dec, gaia_filtered.phot_g_mean_mag,
    gaia_filtered.parallax, gaia_filtered.bp_rp,
    xmatch.original_ext_source_id
FROM
    (SELECT source_id, ra, dec, phot_g_mean_mag, parallax, bp_rp
     FROM gaiadr3.gaia_source
     WHERE CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', 132.825, 11.800, 1)) = 1
     AND phot_g_mean_mag < 14
    ) AS gaia_filtered
JOIN
    gaiadr3.tmass_psc_xsc_best_neighbour AS xmatch
    ON gaia_filtered.source_id = xmatch.source_id
"""

job1 = Gaia.launch_job_async(query_stage1)
stage1_results = job1.get_results().to_pandas()
print("Stage 1 rows:", len(stage1_results))
print(stage1_results.columns.tolist())

INFO: Query finished. [astroquery.utils.tap.core]
Stage 1 rows: 1018
['source_id', 'ra', 'dec', 'phot_g_mean_mag', 'parallax', 'bp_rp', 'original_ext_source_id']


In [24]:
tmass_ids = stage1_results['original_ext_source_id'].dropna().astype(str).tolist()
id_list_str = ",".join(f"'{i}'" for i in tmass_ids)

query_2mass = f"""
SELECT designation, j_m, h_m, ks_m, ph_qual
FROM gaiadr1.tmass_original_valid
WHERE designation IN ({id_list_str})
"""

job2 = Gaia.launch_job_async(query_2mass)
tmass_results = job2.get_results().to_pandas()
print("2MASS rows:", len(tmass_results))

INFO: Query finished. [astroquery.utils.tap.core]
2MASS rows: 1013


In [25]:
final = stage1_results.merge(
    tmass_results, left_on='original_ext_source_id', right_on='designation'
)

print("Final crossmatched star count:", len(final))

print(f"Number of matched stars: {len(final)}")

Final crossmatched star count: 1018
Number of matched stars: 1018


2.2 

In [26]:
print(final['ph_qual'].value_counts())

ph_qual
AAA    997
UAA      5
EEE      2
AAF      2
AAE      2
AUA      2
UUA      1
DCD      1
AEA      1
UAE      1
ADA      1
DAD      1
UUE      1
UAU      1
Name: count, dtype: int64


In [28]:
good_photometry = final['ph_qual'] == 'AAA'
good_parallax = final['parallax'] > 0

clean = final[good_photometry & good_parallax]

print(f"Stars before cuts: {len(final)}")
print(f"Stars with ph_qual == 'AAA': {good_photometry.sum()}")
print(f"Stars with parallax > 0: {good_parallax.sum()}")
print(f"Stars remaining after both cuts: {len(clean)}")
print(f"Number of stars after quality cuts: {len(clean)}")

Stars before cuts: 1018
Stars with ph_qual == 'AAA': 997
Stars with parallax > 0: 1009
Stars remaining after both cuts: 988
Number of stars after quality cuts: 988
